# 01 — Getting started

What `better-calendar` replaces, in one line: the `while d.weekday() > 4: d += timedelta(days=1)`
loop everyone has rewritten three times, and which is wrong the moment a holiday turns up.

The mental model fits in a sentence: **a calendar is a sorted `int64` array of good days**
over a bounded horizon. Membership, offsets, counting and set algebra all reduce to a
`searchsorted` on that array.

In this notebook:

- the three functions you use ninety per cent of the time;
- type transparency — what goes in comes back out;
- the ISDA roll conventions;
- bounds, and why they raise;
- vectorisation, measured.

In [1]:
from datetime import date, datetime, timedelta

import numpy as np
import pandas as pd

import better_calendar as bcal

bcal.__version__

'1.0.0'

## 1. The three everyday functions

`adjust` normalises, `offset` moves, `count` counts. All three take a calendar as the
keyword `cal=`: a `Calendar` object, an identifier, or `None` for the plain `weekday`
calendar (Monday–Friday, no holidays).

In [2]:
# 2026-08-01 is a Saturday.
print("adjust  :", bcal.adjust("2026-08-01"))                    # next business day
print("offset  :", bcal.offset("2026-07-31", 5))                 # five business days later
print("count   :", bcal.count("2026-07-27", "2026-08-01"))       # half-open interval

adjust  : 2026-08-03
offset  : 2026-08-07
count   : 5


With a real market calendar the difference is immediate — 3 July 2026 is a holiday on the
NYSE, Independence Day observed on the Friday:

In [3]:
for name in ("weekday", "XNYS", "fin:TARGET2"):
    print(f"{name:14s} 2026-07-02 + 1 business day -> {bcal.offset('2026-07-02', 1, cal=name)}")

weekday        2026-07-02 + 1 business day -> 2026-07-03
XNYS           2026-07-02 + 1 business day -> 2026-07-06
fin:TARGET2    2026-07-02 + 1 business day -> 2026-07-03


## 2. Type transparency

Whatever goes in comes back out in the same type. No surprise `pd.Timestamp` in the middle
of a pipeline of `date`, no string quietly turned into an object.

In [4]:
inputs = [
    date(2026, 7, 31),
    datetime(2026, 7, 31, 9, 30),
    pd.Timestamp("2026-07-31 09:30"),
    np.datetime64("2026-07-31"),
    "2026-07-31",
    "20260731",
    20260731,
]

rows = []
for value in inputs:
    result = bcal.offset(value, 1)
    rows.append(
        {
            "input type": type(value).__name__,
            "value": repr(value),
            "result": repr(result),
            "output type": type(result).__name__,
        }
    )
pd.DataFrame(rows)

,input type,value,result,output type
0,date,"datetime.date(2026, 7, 31)","datetime.date(2026, 8, 3)",date
1,datetime,"datetime.datetime(2026, 7, 31, 9, 30)","datetime.datetime(2026, 8, 3, 9, 30)",datetime
2,Timestamp,Timestamp('2026-07-31 09:30:00'),Timestamp('2026-08-03 09:30:00'),Timestamp
3,datetime64,np.datetime64('2026-07-31'),np.datetime64('2026-08-03'),datetime64
4,str,'2026-07-31','2026-08-03',str
5,str,'20260731','20260803',str
6,int,20260731,20260803,int


Two things worth noting:

- the `datetime` at 09:30 comes back **at 09:30**: an offset only touches the date part;
- the integer is read as `yyyymmdd`, never as a Unix timestamp.

Parsing is deliberately strict. Ambiguous formats are refused rather than guessed:

In [5]:
for text in ("31/07/2026", "07/31/2026", "Jul 31 2026"):
    try:
        bcal.to_date(text)
    except bcal.BetterCalendarError as exc:
        print(f"{text!r:16s} -> {str(exc)[:95]}…")

'31/07/2026'     -> Cannot parse date string '31/07/2026'. Accepted formats are ISO-8601 ('2026-07-31', '2026-07-31…
'07/31/2026'     -> Cannot parse date string '07/31/2026'. Accepted formats are ISO-8601 ('2026-07-31', '2026-07-31…
'Jul 31 2026'    -> Cannot parse date string 'Jul 31 2026'. Accepted formats are ISO-8601 ('2026-07-31', '2026-07-3…


In [6]:
# A Unix timestamp passed by accident does not silently become a date.
try:
    bcal.to_date(1785456000)
except bcal.BetterCalendarError as exc:
    print(exc)

Cannot interpret 1785456000 as a date. Ints are read as yyyymmdd (for example 20260731); this guards against Unix timestamps being passed by accident. Pass a datetime.date or an ISO-8601 string instead.


## 3. Roll conventions

Seven conventions, reachable by full name or by short ISDA alias (`"F"`, `"MF"`, `"P"`,
`"MP"`, `"N"`), case-insensitive.

The table below applies all of them to three awkward dates: an ordinary Saturday, and
**Sunday 31 May 2026** — the last day of its month, which is the whole difference between
`FOLLOWING` and `MODIFIED_FOLLOWING`.

In [7]:
dates = ["2026-08-01", "2026-05-31", "2026-02-01"]
table = {}
for roll in bcal.Roll:
    row = {}
    for day in dates:
        try:
            row[day] = bcal.adjust(day, roll)
        except bcal.BetterCalendarError as exc:
            row[day] = f"<{type(exc).__name__}>"
    table[roll.value] = row

result = pd.DataFrame(table).T
result.columns = [f"{c} ({date.fromisoformat(c).strftime('%a')})" for c in result.columns]
result

,2026-08-01 (Sat),2026-05-31 (Sun),2026-02-01 (Sun)
none,2026-08-01,2026-05-31,2026-02-01
following,2026-08-03,2026-06-01,2026-02-02
preceding,2026-07-31,2026-05-29,2026-01-30
modified_following,2026-08-03,2026-05-29,2026-02-02
modified_preceding,2026-08-03,2026-05-29,2026-02-02
nearest,2026-07-31,2026-06-01,2026-02-02
raise,<NotABusinessDayError>,<NotABusinessDayError>,<NotABusinessDayError>


How to read the two interesting columns:

- **31 May** (Sunday, month end): `following` leaves May for 1 June, so
  `modified_following` turns back to Friday the 29th.
- **1 February** (Sunday, month start): `preceding` leaves February for 30 January, so
  `modified_preceding` goes forward to Monday the 2nd.

That is exactly why the "modified" variants exist, and exactly where hand-rolled
implementations get it wrong.

In [8]:
# Roll.RAISE refuses instead of adjusting — useful for validating an input.
try:
    bcal.adjust("2026-08-01", bcal.Roll.RAISE)
except bcal.NotABusinessDayError as exc:
    print(exc)

2026-08-01 is not a business day in calendar 'weekday' and roll=Roll.RAISE forbids adjusting it. Pass a different roll convention (for example Roll.MODIFIED_FOLLOWING) to move it to a nearby business day.


## 4. Counting and intervals

The half-open interval `[start, end)` is the default **everywhere**. Any other convention
has to be asked for by name.

In [9]:
start, end = "2026-07-27", "2026-07-31"   # Monday -> Friday
pd.DataFrame(
    [
        {"closed": c, "count": bcal.count(start, end, closed=c)}
        for c in ("left", "right", "both", "neither")
    ]
).set_index("closed")

,count
closed,
left,4
right,4
both,5
neither,3


Counting is **signed**, which is what makes `count(d, offset(d, n)) == n` hold for negative
`n` too:

In [10]:
origin = "2026-07-31"
for n in (-10, -1, 0, 1, 10):
    arrival = bcal.offset(origin, n)
    assert bcal.count(origin, arrival) == n
    print(f"offset({origin}, {n:>3}) = {arrival}   count -> {bcal.count(origin, arrival):>3}")

offset(2026-07-31, -10) = 2026-07-17   count -> -10
offset(2026-07-31,  -1) = 2026-07-30   count ->  -1
offset(2026-07-31,   0) = 2026-07-31   count ->   0
offset(2026-07-31,   1) = 2026-08-03   count ->   1
offset(2026-07-31,  10) = 2026-08-14   count ->  10


`DateRange` is the small value object that goes with it:

In [11]:
period = bcal.DateRange("2026-07-27", "2026-08-03")
print("length            :", len(period))
print("3 August inside?  :", date(2026, 8, 3) in period)      # half-open: no
print("business days     :", len(period.business_days()))
print("24/7 days         :", len(period.business_days("crypto:24x7")))

quarter = bcal.DateRange("2026-01-01", "2026-04-01")
print("\nmonthly split     :", [str(p.start) for p in quarter.split("M")])
print("the pieces tile without gaps:",
      sum(len(p) for p in quarter.split("M")) == len(quarter))

length            : 7
3 August inside?  : False
business days     : 5
24/7 days         : 7

monthly split     : ['2026-01-01', '2026-02-01', '2026-03-01']
the pieces tile without gaps: True


## 5. Bounds: never extrapolate

Every calendar has a finite, explicit horizon. Leaving it raises an error that says
**where** the bounds are — never an invented answer.

In [12]:
# The default horizon is driven by a single constant, never by a scattered literal.
print("default horizon :", bcal.MIN_YEAR, "->", bcal.MAX_YEAR)
print("weekday bounds  :", bcal.get("weekday").bounds)

default horizon : 1970 -> 2100
weekday bounds  : (datetime.date(1970, 1, 1), datetime.date(2100, 12, 31))


In [13]:
tokyo = bcal.get("XTKS")
print("XTKS bounds :", tokyo.bounds)

try:
    tokyo.is_bday("1996-12-31")
except bcal.OutOfBoundsError as exc:
    print("\n" + str(exc))

XTKS bounds : (datetime.date(1997, 1, 1), datetime.date(2100, 12, 31))

1996-12-31 is outside the bounds of calendar 'XTKS' (1997-01-01 to 2100-12-31, inclusive). Rebuild the calendar with wider `bounds`, or raise MAX_YEAR in better_calendar.core.epoch.


Those bounds are not decoration: they reflect what the upstream source can actually answer.
`exchange-calendars` refuses to evaluate Tokyo before 1997 or Hong Kong after 2049, and
QuantLib's lunar calendars stop where their tables stop.

In [14]:
pd.DataFrame(
    [
        {"calendar": n, "from": bcal.get(n).bounds[0], "to": bcal.get(n).bounds[1]}
        for n in ("weekday", "XNYS", "XTKS", "XHKG", "ql:Israel.TASE", "ql:China.SSE")
    ]
).set_index("calendar")

,from,to
calendar,,
weekday,1970-01-01,2100-12-31
XNYS,1970-01-01,2100-12-31
XTKS,1997-01-01,2100-12-31
XHKG,1970-01-01,2049-12-31
ql:Israel.TASE,1970-01-01,2025-12-31
ql:China.SSE,1970-01-01,2026-12-31


## 6. Vectorisation

Every scalar function also takes an array. The gain is not algorithmic — the scalar path is
already a `searchsorted`, so O(log n) — but in not paying for Python-level iteration and
type conversion on every element.

In [15]:
import time

days = pd.date_range("2020-01-01", "2026-12-31", freq="D")
calendar = bcal.get("XNYS")

t0 = time.perf_counter()
vectorised = calendar.offset(days, 5)
elapsed_vec = time.perf_counter() - t0

t1 = time.perf_counter()
one_by_one = [calendar.offset(d, 5) for d in days[:2000]]
elapsed_loop = (time.perf_counter() - t1) * len(days) / 2000

print(f"{len(days)} dates")
print(f"  vectorised  : {elapsed_vec * 1000:8.1f} ms")
print(f"  one by one  : {elapsed_loop * 1000:8.1f} ms  (extrapolated)")
print(f"  ratio       : {elapsed_loop / elapsed_vec:8.0f}x")

2557 dates
  vectorised  :      0.8 ms
  one by one  :     35.9 ms  (extrapolated)
  ratio       :       42x


## Recap

| Function | Role |
|---|---|
| `adjust(d, roll, cal=)` | move a date onto a business day |
| `offset(d, n, cal=)` | move by `n` business days |
| `count(a, b, cal=)` | count business days, signed |
| `is_bday`, `next_bday`, `prev_bday` | membership and neighbours |
| `to_date`, `to_datetime`, `to_timestamp` | strict conversions |
| `DateRange` | interval, iteration, splitting |
| `Roll` | the seven ISDA conventions |

**Next:** [02 — Calendars and algebra](02-calendars-and-algebra.ipynb)